In [1]:
import torch

from torch_openreml import MarginalREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix, CovariancePropagation, Sum

n, p = 50, 2

y = torch.randn(n)
X = torch.randn(n, p)

Z = DummyMatrix(["a", "b"] * 25)

V = Sum(
    CovariancePropagation(
        Z,
        ScalarMatrix(2),
    ),
    ScalarMatrix(n),
)

reml = MarginalREML(V)

theta_start = torch.zeros(V.num_free_params)

theta_hat, beta_hat, n_iter = reml.optimize(
    y,
    X,
    theta_start,
    verbose=1,
)

[  1] ∥∇∥:      17.3593, ∥Δ∥: 7.1602, η: 1.00, ∥Δᶜ∥: 7.1602, log 𝓛: -21.7135
[  2] ∥∇∥:      10.6130, ∥Δ∥: 0.0905, η: 1.00, ∥Δᶜ∥: 0.0905, log 𝓛: -18.9849 (+2.7286)
[  3] ∥∇∥:       0.9054, ∥Δ∥: 0.0093, η: 1.00, ∥Δᶜ∥: 0.0093, log 𝓛: -18.4767 (+0.5082)
[  4] ∥∇∥:       0.0083, ∥Δ∥: 0.0001, η: 1.00, ∥Δᶜ∥: 0.0001, log 𝓛: -18.4725 (+0.0042)
[  5] ∥∇∥:       0.0001, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, log 𝓛: -18.4725 (+0.0000)

[∇: score, Δ: 𝐉⁻¹∇, η: learning rate, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.00s



In [2]:
theta_last = reml.get_theta(select="last")
theta_best = reml.get_theta(select="best")

beta_last = reml.get_beta(select="last")
beta_best = reml.get_beta(select="best")

In [3]:
beta_hat = reml.blue(y, X, theta_hat)

In [4]:
y_hat = reml.predict(
    y,
    X,
    theta_hat,
)

In [5]:
e = reml.residual(
    y,
    X,
    theta_hat,
)

In [6]:
loglik = reml.loglik(y, X, theta_hat)

In [7]:
import torch

from torch_openreml import MarginalREML

from torch_openreml.covariance import (
    Augment,
    BlockDiagonal,
    DummyMatrix,
    IdentityMatrix,
    ScalarMatrix,
    Sum,
    CovariancePropagation,
    KroneckerProduct,
)

from torch_openreml.example_data import john_alpha

# --- response ---
y = torch.tensor(john_alpha["yield"].values)

# --- fixed effects ---
X = Augment(
    torch.ones(len(john_alpha), 1),
    DummyMatrix(john_alpha["rep"], drop_first=True)
)()

# --- random effects ---
gen = john_alpha["gen"]
rep = john_alpha["rep"]
block = john_alpha["block"]
n = len(john_alpha)

# --- marginal covariance ---
V = Sum(
    CovariancePropagation(
        Augment(
            DummyMatrix(gen),
            DummyMatrix(rep, block),
        ),
        BlockDiagonal(
            ScalarMatrix(gen),
            KroneckerProduct(
                IdentityMatrix(rep),
                ScalarMatrix(block),
            ),
        ),
    ),
    ScalarMatrix(n),
)

# --- REML fit ---
reml = MarginalREML(V)

theta_start = torch.zeros(V.num_free_params)

theta_hat, beta_hat, n_iter = reml.optimize(
    y,
    X,
    theta_start,
    verbose=1,
)

# --- results ---
print("theta:", theta_hat)

print("variance components:", V.build_params(theta_hat))

print("fixed effects:", beta_hat)

print("loglik:", reml.loglik(y, X, theta_hat))

[  1] ∥∇∥:      41.8197, ∥Δ∥: 8.9438, η: 1.00, ∥Δᶜ∥: 8.9438, log 𝓛: -34.3129
[  2] ∥∇∥:  315236.5000, ∥Δ∥: 0.8838, η: 1.00, ∥Δᶜ∥: 0.8838, log 𝓛: -201188.2344 (-201153.9215)
[  3] ∥∇∥:  119229.1953, ∥Δ∥: 0.8520, η: 1.00, ∥Δᶜ∥: 0.8520, log 𝓛: -71844.9922 (+129343.2422)
[  4] ∥∇∥:   44277.3672, ∥Δ∥: 0.8143, η: 1.00, ∥Δᶜ∥: 0.8143, log 𝓛: -26066.7031 (+45778.2891)
[  5] ∥∇∥:   16331.1357, ∥Δ∥: 0.7587, η: 1.00, ∥Δᶜ∥: 0.7587, log 𝓛: -9457.5859 (+16609.1172)
[  6] ∥∇∥:    6001.6343, ∥Δ∥: 0.7142, η: 1.00, ∥Δᶜ∥: 0.7142, log 𝓛: -3393.0317 (+6064.5542)
[  7] ∥∇∥:    2194.9788, ∥Δ∥: 0.7031, η: 1.00, ∥Δᶜ∥: 0.7031, log 𝓛: -1181.3386 (+2211.6931)
[  8] ∥∇∥:     793.8171, ∥Δ∥: 0.6952, η: 1.00, ∥Δᶜ∥: 0.6952, log 𝓛: -382.5117 (+798.8269)
[  9] ∥∇∥:     278.6996, ∥Δ∥: 0.6714, η: 1.00, ∥Δᶜ∥: 0.6714, log 𝓛: -102.3060 (+280.2057)
[ 10] ∥∇∥:      90.4162, ∥Δ∥: 0.5957, η: 1.00, ∥Δᶜ∥: 0.5957, log 𝓛: -11.4293 (+90.8767)
[ 11] ∥∇∥:      23.9234, ∥Δ∥: 0.4006, η: 1.00, ∥Δᶜ∥: 0.4006, log 𝓛:  12.6791 (+24.1085)
[ 12]

In [8]:
scores = [
    torch.norm(s).item()
    for s in reml.history["score"]
]

logliks = [
    ll.item()
    for ll in reml.history["loglik"]
]

print(
    "Score norms:",
    [f"{s:.6f}" for s in scores],
)

print(
    "Log-likelihoods:",
    [f"{ll:.4f}" for ll in logliks],
)

Score norms: ['41.819733', '315236.500000', '119229.195312', '44277.367188', '16331.135742', '6001.634277', '2194.978760', '793.817139', '278.699646', '90.416214', '23.923351', '3.867782', '0.244802', '0.010348', '0.000439', '0.000039']
Log-likelihoods: ['-34.3129', '-201188.2344', '-71844.9922', '-26066.7031', '-9457.5859', '-3393.0317', '-1181.3386', '-382.5117', '-102.3060', '-11.4293', '12.6791', '16.5908', '16.8078', '16.8097', '16.8097', '16.8098']
